In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, Any
import numpy as np

from recsysconfident.data_handling.datasets.datasetinfo import DatasetInfo
from recsysconfident.data_handling.datasets.csv_reader import CsvReader

plt.rcParams.update({
    "font.size": 26,
    "axes.titlesize": 26,
    "axes.labelsize": 26,
    "xtick.labelsize": 26,
    "ytick.labelsize": 26,
})

def plot_user_item_distributions(df, info, bins=50):

    df.drop_duplicates(subset=[info.user_col, info.item_col], inplace=True)
    user_counts = df[info.user_col].value_counts()
    item_counts = df[info.item_col].value_counts()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # User distribution (weighted by number of interactions)
    axes[0].hist(
        user_counts,
        bins=bins,
        weights=user_counts
    )
    axes[0].set_title("User Interaction Distribution")
    axes[0].set_xlabel("Interactions per User")
    axes[0].set_ylabel("Number of Interactions")

    # Item distribution (weighted by number of interactions)
    axes[1].hist(
        item_counts,
        bins=bins,
        weights=item_counts
    )
    axes[1].set_title("Item Interaction Distribution")
    axes[1].set_xlabel("Interactions per Item")
    axes[1].set_ylabel("Number of Interactions")

    plt.tight_layout()
    plt.show()

def plot_dataset_distributions(
    df_before: pd.DataFrame, #before preprocessing
    df_after: pd.DataFrame, #after preprocessing
    info,
    name,
    bins=50,
    log_scale=True
):
    names = ("Before", "After")
    datasets = [df_before, df_after]

    items_per_user = [
        df.groupby(info.user_col)[info.item_col].count()
        for df in datasets
    ]

    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 5))
    fig.tight_layout(pad=4.0)

    for i in range(2):
        axes[i].hist(items_per_user[i], bins=bins)
        axes[i].set_title(f"Items per user ({names[i]})")
        axes[i].set_xlabel("Number of items")
        axes[i].set_ylabel("Users")
        if log_scale:
            axes[i].set_yscale("log")

    plt.tight_layout()
    plt.savefig(f"./plots/{name}-dist-items-per-user.eps", bbox_inches="tight")
    plt.show()

def plot_after_distributions(data: Dict[str, Dict[str, Any]], bins: int = 50):
    n_cols = len(data)
    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4), squeeze=False)

    for col_idx, (dataset, ds_data) in enumerate(data.items()):
        ax = axes[0, col_idx]
        df_after = ds_data["after"]
        rating_col = ds_data["info"].relevance_col

        values = df_after[rating_col].dropna()
        ax.hist(values, bins=bins, color="#4682b4", edgecolor="white")
        
        ax.set_title(dataset)
        ax.set_xlabel(rating_col)
        if col_idx == 0:
            ax.set_ylabel("Frequency")

    plt.tight_layout()
    plt.show()



In [ ]:
processed_dfs_names = ['amazon-movies-tvs', 'jester-joke', 'ml-1m']

datasets = {}

for dataset_name in processed_dfs_names:
    base_uri = f"../runs/data_splits/{dataset_name}/0/"
    fit_df = pd.read_csv(f"{base_uri}/ratings.fit.csv")
    eval_df = pd.read_csv(f"{base_uri}/ratings.val.csv")
    test_df = pd.read_csv(f"{base_uri}/ratings.test.csv")
    after_df = pd.concat([fit_df, eval_df, test_df])

    df_info = DatasetInfo(**json.load(open(f'../data/{dataset_name}/info.json', 'r')),
                        database_name=dataset_name,
                        root_uri="..")
    before_df = CsvReader(df_info).read()
    
    datasets[dataset_name] = {
        "before": before_df,
        "after": after_df,
        "info": df_info
    }

In [ ]:
for dataset_name in datasets.keys():
    before_df = datasets[dataset_name]['before']
    after_df = datasets[dataset_name]['after']
    df_info = datasets[dataset_name]['info']
    plot_dataset_distributions(before_df, after_df, df_info, dataset_name)

In [ ]:
for dataset_name in datasets.keys():
    before_df = datasets[dataset_name]['before']
    after_df = datasets[dataset_name]['after']
    df_info = datasets[dataset_name]['info']
    print(dataset_name)
    plot_user_item_distributions(after_df, df_info)

In [ ]:
plot_after_distributions(datasets)

In [6]:
n_bins = len(datasets['jester-joke']['after']['rating'].unique())
min_value = datasets['jester-joke']['after']['rating'].min()
max_value = datasets['jester-joke']['after']['rating'].max()
ratings = datasets['jester-joke']['after']['rating'].values

In [14]:
import torch

def fit_approx_polynomial(ratings, min_value, max_value, degree=100, bins='auto'):
    counts, edges = np.histogram(ratings, bins=bins, range=(min_value, max_value), density=True)
    centers = (edges[:-1] + edges[1:]) / 2
    coeffs = np.polyfit(centers, counts, deg=degree)
    return np.poly1d(coeffs)
    

def get_y(poly_model, x):

    y = poly_model(x)
    y[y < 0] = 0
    return y

def get_density(ratings, n_values, min_val, max_val):
    counts = torch.histc(ratings.float(), bins=n_values, min=min_val, max=max_val)
    density = counts / counts.sum()
    return ratings.unique(), density.view(-1, 1)


In [ ]:
poly_model = fit_approx_polynomial(torch.from_numpy(ratings), min_value, max_value, degree=20)

x_plot = np.linspace(min_value, max_value, 500)
y_plot = poly_model(x_plot)

plt.hist(ratings, bins='auto', range=(min_value, max_value), density=True, alpha=0.5)
plt.plot(x_plot, y_plot, color='red')

plt.show()